<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week7/Day3/ExerciseXP/Exercises_XP_RAG_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [ ]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

In [ ]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
# CORRECTION : RetrievalQA s'importe depuis la bibliothèque centrale de chaînes de LangChain
from langchain.chains import RetrievalQA


## 1) Load dataset and convert to Documents


In [ ]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

documents: List[Document] = []
for i, row in enumerate(ds):
    # TODO: instancier correctement l'objet Document avec son contenu et ses métadonnées
    documents.append(
        Document(
            page_content=str(row[text_column]),
            metadata={"source": str(row[source_column])}
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])


## 2) Split into chunks


In [ ]:
# TODO: Spécifier la taille des morceaux et le chevauchement sémantique
chunk_size = 500       # Taille optimale pour conserver une bonne granularité d'information
chunk_overlap = 50     # Chevauchement de 10% pour ne pas perdre d'information aux frontières

# TODO: Initialiser le splitter de texte récursif avec les hyperparamètres
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", " ", ""] # Découpage intelligent par paragraphes puis phrases
)

splits = splitter.split_documents(documents)
print("Chunks:", len(splits))
print("First chunk:", splits[0].metadata)
print(splits[0].page_content[:350])


## 3) Vector store + retriever (FAISS)


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

# TODO: Renseigner les fragments de texte, le modèle d'embedding et la stratégie Cosinus
vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.COSINE
)

# Configuration du moteur de recherche pour extraire le top k=4 fragments pertinents
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")


Retriever ready


## 4) Build the RAG chain


In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
# CORRECTION SIRA LABS : Importation depuis le chemin d'accès stable et centralisé de LangChain
from langchain.chains import RetrievalQA

llm_id = "google/flan-t5-small"

# TODO: Configurer le pipeline Hugging Face pour une tâche de type text2text-generation
hf = pipeline(
    task="text2text-generation",
    model=llm_id,
    max_new_tokens=100,
    model_kwargs={"torch_dtype": "float32"},
    device_map="auto"  # Utilisation automatique du GPU si disponible
)

llm = HuggingFacePipeline(pipeline=hf)

# TODO: Assembler la chaîne de Questions-Réponses avec les paramètres exigés
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",  # Injection simple des fragments de texte dans le prompt
    return_source_documents=True  # Optionnel mais recommandé pour afficher les sources
)

print("RAG chain ready")


## 5) Demo: RAG vs no-RAG


In [ ]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# No-RAG (LLM only)
no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

# RAG
# TODO: Passer la question à la chaîne RAG dans le format dictionnaire approprié
rag_result = qa.invoke({"query": q})

print("Q:", q)
print("\nNo-RAG answer:\n", no_rag_answer)
print("\nRAG answer:\n", rag_result["result"])
print("\nSources:")
# Affichage automatique des citations de documents sources pour le débogage
for d in rag_result["source_documents"]:
    print("-", d.metadata.get("source"))
